# 03 - Mainnet CLMM

Concentrated-liquidity AMM analysis. This notebook is intentionally separate from CPMM because CLMM needs tick/active-liquidity state and cannot reuse the CPMM closed-form baseline.

Current inputs:
- `results/historical_clmm_decoded.csv` — decoded historical CLMM swap observations
- `results/historical_clmm_swaps_status.csv` — inclusion/exclusion ledger
- `results/historical_clmm_pipeline_summary.csv` — collection/decode funnel

Future input:
- `results/historical_clmm_candidates.csv` — counterfactual CLMM sandwich results after historical tick-array pre-state and replay validation exist


## Load data


In [1]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "helpers").exists():
        sys.path.insert(0, str(candidate))
        break
    if (candidate / "notebooks" / "helpers").exists():
        sys.path.insert(0, str(candidate / "notebooks"))
        break

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from helpers import (
    best_conditions,
    blocker_table,
    data_readiness,
    find_repo_root,
    historical_counterfactual_summary,
    hypothesis_scorecard,
    load_inputs,
    plot_realized_heatmap,
    plot_sensitivity_lines,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

ROOT = find_repo_root()
inputs = load_inputs(ROOT)

clmm = inputs["frames"]["historical_clmm"]
clmm_decoded = inputs["frames"]["historical_clmm_decoded"]
clmm_status = inputs["frames"]["historical_clmm_swaps_status"]
clmm_summary = inputs["frames"]["historical_clmm_pipeline_summary"]
clmm_state_probe = inputs["frames"]["historical_clmm_state_probe"]
cpmm = inputs["frames"]["historical_cpmm"]

display(data_readiness(ROOT, inputs, [
    "historical_clmm_decoded",
    "historical_clmm_swaps_status",
    "historical_clmm_pipeline_summary",
    "historical_clmm_state_probe",
    "historical_clmm",
    "historical_cpmm",
]))


,dataset,file,rows,columns,ready_for_final_numbers,note
0,historical_clmm_decoded,results/historical_clmm_decoded.csv,4,29,False,decode coverage only; not profitability evidence
1,historical_clmm_swaps_status,results/historical_clmm_swaps_status.csv,50,16,True,ok
2,historical_clmm_pipeline_summary,results/historical_clmm_pipeline_summary.csv,2,7,True,ok
3,historical_clmm_state_probe,results/historical_clmm_state_probe.csv,4,22,False,state requirements only; not profitability evi...
4,historical_clmm,missing,0,0,False,missing file
5,historical_cpmm,results/historical_cpmm_candidates.csv,72,42,False,legacy schema: regenerate CSV


## Scope check


In [2]:
if not clmm.empty:
    display(historical_counterfactual_summary(clmm))
elif not clmm_decoded.empty:
    display(Markdown(
        f"Decoded `{len(clmm_decoded)}` CLMM swap observation(s), but no counterfactual candidate CSV exists yet. "
        "This is expected until historical PoolState/tick-array pre-state and CLMM replay validation are implemented."
    ))
    display(clmm_summary)
    if not clmm_state_probe.empty:
        display(clmm_state_probe[[
            "pool_label",
            "slot",
            "instruction_index",
            "required_account_count",
            "current_probe_slot",
            "current_pool_state_ok",
            "current_amm_config_ok",
            "current_tick_arrays_ok",
            "historical_state_available",
            "candidate_ready",
        ]].head(20))
    display(
        clmm_decoded[[
            "pool_label",
            "slot",
            "signature",
            "instruction_index",
            "swap_variant",
            "direction",
            "amount_in",
            "actual_amount_out",
            "historical_state_status",
        ]].head(20)
    )
else:
    display(Markdown(
        "No decoded CLMM observations exist yet. Run `cargo run -p fork --bin historical_clmm -- --pool clmm_wsol_usdc --limit-per-pool 50 run-all`."
    ))
    if not clmm_status.empty:
        display(clmm_status["analysis_status"].value_counts().rename_axis("analysis_status").reset_index(name="rows"))


Decoded `4` CLMM swap observation(s), but no counterfactual candidate CSV exists yet. This is expected until historical PoolState/tick-array pre-state and CLMM replay validation are implemented.

,stage,pool_label,input_rows,output_rows,rejected_rows,top_rejection_reason,created_at_unix
0,collect_signatures,clmm_wsol_usdc,0,50,0,NaN,1778340744
1,build_decoded,clmm_wsol_usdc,50,4,46,tx_failed (46),1778340744


,pool_label,slot,instruction_index,required_account_count,current_probe_slot,current_pool_state_ok,current_amm_config_ok,current_tick_arrays_ok,historical_state_available,candidate_ready
0,clmm_wsol_usdc,418638335,3001,7,418642660,True,True,3,False,False
1,clmm_wsol_usdc,418638335,4000,7,418642660,True,True,3,False,False
2,clmm_wsol_usdc,418638334,3000,7,418642660,True,True,3,False,False
3,clmm_wsol_usdc,418638334,2000,7,418642660,True,True,3,False,False


,pool_label,slot,signature,instruction_index,swap_variant,direction,amount_in,actual_amount_out,historical_state_status
0,clmm_wsol_usdc,418638335,3iA2sMXAPbZ3YitpKejiWCMXg2fUPyCMSeQLrmeFoaWVyZ...,3001,swap,base_input,400000000,4311696696,missing_tick_array_pre_state: getTransaction d...
1,clmm_wsol_usdc,418638335,282NzuP3svhWFv175y3eLV6zwLZz8ARfS26qHo4P872AgR...,4000,swap,base_input,244682478,2637533509,missing_tick_array_pre_state: getTransaction d...
2,clmm_wsol_usdc,418638334,3BUYTx9xQWwsFzFmbcXTRBavCUvZTb1imLJ7oddVsGJN9e...,3000,swap,base_input,198077413,2135181573,missing_tick_array_pre_state: getTransaction d...
3,clmm_wsol_usdc,418638334,5okuCT6Nk244dDueNUCEBBe93K1t9cQAZhZS3gErBg1qv3...,2000,swap,base_input,367536829,3961929825,missing_tick_array_pre_state: getTransaction d...


## Required CLMM candidate fields


In [3]:
required = pd.DataFrame([
    {"field": "pool_type", "why": "distinguish raydium_clmm/orca_whirlpool from cpmm"},
    {"field": "pool_label, pool_address", "why": "group results by selected pool"},
    {"field": "slot, signature, instruction_index", "why": "trace every counterfactual row back to a historical swap"},
    {"field": "amount_in, min_amount_out, actual_amount_out", "why": "victim size and slippage bound"},
    {"field": "sqrt_price_x64_before, liquidity_before, tick_current_before", "why": "CLMM state at victim pre-state"},
    {"field": "tick_arrays_before", "why": "needed for executable CLMM swap simulation across ticks"},
    {"field": "fee_rate, protocol_fee_rate", "why": "fee-aware profitability"},
    {"field": "tx_cost_per_leg", "why": "net profit, not just gross extraction"},
])
display(required)


,field,why
0,pool_type,distinguish raydium_clmm/orca_whirlpool from cpmm
1,"pool_label, pool_address",group results by selected pool
2,"slot, signature, instruction_index",trace every counterfactual row back to a histo...
3,"amount_in, min_amount_out, actual_amount_out",victim size and slippage bound
4,"sqrt_price_x64_before, liquidity_before, tick_...",CLMM state at victim pre-state
5,tick_arrays_before,needed for executable CLMM swap simulation acr...
6,"fee_rate, protocol_fee_rate",fee-aware profitability
7,tx_cost_per_leg,"net profit, not just gross extraction"


## CPMM vs CLMM comparison


In [4]:
if clmm.empty or cpmm.empty:
    display(Markdown("Counterfactual CPMM vs CLMM comparison waits for both historical CPMM and CLMM candidate outputs."))
    if not clmm_decoded.empty:
        display(Markdown("CLMM decode coverage is available, but it is not profitability evidence yet."))
else:
    cpmm_summary = historical_counterfactual_summary(cpmm).assign(family="CPMM")
    clmm_candidate_summary = historical_counterfactual_summary(clmm).assign(family="CLMM")
    display(pd.concat([cpmm_summary, clmm_candidate_summary], ignore_index=True))


Counterfactual CPMM vs CLMM comparison waits for both historical CPMM and CLMM candidate outputs.

CLMM decode coverage is available, but it is not profitability evidence yet.

## Thesis-ready takeaways


In [5]:
if clmm.empty:
    lines = [
        "- CLMM profitability remains an empirical gap, not a result.",
        f"- Decoded CLMM observations available: `{len(clmm_decoded)}` rows.",
        f"- CLMM state probe rows available: `{len(clmm_state_probe)}` rows.",
        "- Next blocker: historical PoolState/tick-array pre-state and victim-swap replay validation.",
    ]
    display(Markdown("\n".join(lines)))
else:
    display(Markdown(f"- Historical CLMM counterfactual candidates loaded: `{len(clmm)}` rows."))


- CLMM profitability remains an empirical gap, not a result.
- Decoded CLMM observations available: `4` rows.
- CLMM state probe rows available: `4` rows.
- Next blocker: historical PoolState/tick-array pre-state and victim-swap replay validation.